In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

In [3]:
X = np.load('X_features.npy')
y = np.load('y_labels.npy')

In [4]:
# CNNs expect a 4D input: (Batch_Size, Height, Width, Channels)
# Since our spectrograms are grayscale, we add a '1' for the channel.
X = X.reshape(X.shape[0], X.shape[1], X.shape[2], 1)

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Loaded {X_train.shape[0]} training samples and {X_test.shape[0]} testing samples.")

✅ Loaded 415 training samples and 104 testing samples.


In [5]:
model = models.Sequential([
    # Layer 1: Catching basic textures
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 313, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # Layer 2: Catching more complex shapes (like the gaps you saw!)
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Layer 3: Flattening the image into a list of features
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    
    # Output: Is it an anomaly?
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

c:\Users\mayank goyal\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 311, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 155, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 153, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 76, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 76, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 145920)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     9,338,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,357,825 (35.70 MB)

 Trainable params: 9,357,825 (35.70 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(X_train, y_train, 
                    epochs=15, 
                    batch_size=32, 
                    validation_data=(X_test, y_test))

Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 395ms/step - accuracy: 0.6747 - loss: 1.5522 - val_accuracy: 0.7692 - val_loss: 0.5237
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 352ms/step - accuracy: 0.6916 - loss: 0.6100 - val_accuracy: 0.7692 - val_loss: 0.5451
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 341ms/step - accuracy: 0.7253 - loss: 0.5851 - val_accuracy: 0.7692 - val_loss: 0.5379
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 335ms/step - accuracy: 0.7253 - loss: 0.5907 - val_accuracy: 0.7692 - val_loss: 0.5073
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 335ms/step - accuracy: 0.7253 - loss: 0.5731 - val_accuracy: 0.7692 - val_loss: 0.5121
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 387ms/step - accuracy: 0.7253 - loss: 0.5453 - val_accuracy: 0.7692 - val_loss: 0.4754
Epoch 7/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 373ms/step - accuracy: 0.7253 - loss: 0.5202 - val_accuracy: 0.7692 - val_loss: 0.4998
Epoch 8/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 378ms/step - accuracy: 0.7253 - loss: 0.5293 - val_accuracy: 0.

In [9]:
# Updated Model Snippet for higher accuracy
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 313, 1)),
    layers.BatchNormalization(), # <-- Add this!
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(), # <-- Add this!
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'), # <-- A third, deeper layer
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 126, 311, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 126, 311, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 63, 155, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 61, 153, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 61, 153, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 30, 76, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 74, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 14, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 66304)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     8,487,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,580,225 (32.73 MB)

 Trainable params: 8,580,033 (32.73 MB)

 Non-trainable params: 192 (768.00 B)

In [10]:
history = model.fit(X_train, y_train, 
                    epochs=30, 
                    batch_size=32, 
                    validation_data=(X_test, y_test))

Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 13s 832ms/step - accuracy: 0.6337 - loss: 10.2603 - val_accuracy: 0.7788 - val_loss: 0.6820
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 756ms/step - accuracy: 0.8096 - loss: 0.7434 - val_accuracy: 0.7692 - val_loss: 0.6686
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 755ms/step - accuracy: 0.8940 - loss: 0.2873 - val_accuracy: 0.5769 - val_loss: 0.6842
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 761ms/step - accuracy: 0.9422 - loss: 0.2073 - val_accuracy: 0.2308 - val_loss: 1.5272
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 756ms/step - accuracy: 0.9446 - loss: 0.1595 - val_accuracy: 0.2308 - val_loss: 3.1315
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 760ms/step - accuracy: 0.9542 - loss: 0.1163 - val_accuracy: 0.2308 - val_loss: 8.2139
Epoch 7/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 764ms/step - accuracy: 0.9735 - loss: 0.0916 - val_accuracy: 0.2308 - val_loss: 10.3804
Epoch 8/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 769ms/step - accuracy: 0.9807 - loss: 0.0456 - val_ac